# 03 — Results analysis

Pulls every run from MLflow and builds the comparison tables and charts of the report.
Set the MLflow environment variables (or a `.env` file) before running.

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from brain_tumor.config import load_params, project_path
from brain_tumor.tracking import setup_mlflow

cfg = load_params()
setup_mlflow(cfg, "custom_cnn")  # connects; the experiment name does not matter here
prefix = cfg["mlflow"]["experiment_prefix"]
OUT = project_path(cfg, "reports_dir")
(OUT / "figures").mkdir(parents=True, exist_ok=True)

runs = mlflow.search_runs(search_all_experiments=True,
                          filter_string="attributes.status = 'FINISHED'")
runs = runs[runs["tags.mlflow.runName"].notna()]
print(len(runs), "finished runs")

## 1. Main comparison table (test set, patient-level split)

In [ ]:
CLASSES = cfg["data"]["class_names"]
cols = {
    "tags.mlflow.runName": "run",
    "tags.model": "model",
    "tags.split_column": "split",
    "metrics.best_val_macro_f1": "val macro-F1",
    "metrics.test_accuracy": "test accuracy",
    "metrics.test_macro_f1": "test macro-F1",
    **{f"metrics.test_recall_{c}": f"recall {c}" for c in CLASSES},
    "metrics.test_n_parameters": "parameters",
    "metrics.test_inference_ms_per_image": "ms / image",
    "metrics.train_minutes": "train min",
}
table = runs[[c for c in cols if c in runs.columns]].rename(columns=cols)
table = table[table["model"].notna()].sort_values("test macro-F1", ascending=False)
main = table[table["split"] == "split"].round(3)
display(main)
main.to_csv(OUT / "comparison.csv", index=False)
(OUT / "comparison.md").write_text(main.to_markdown(index=False))

## 2. Accuracy vs. cost

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for _, r in main.iterrows():
    ax.scatter(r["parameters"] / 1e6, r["test macro-F1"], s=60)
    ax.annotate(r["run"], (r["parameters"] / 1e6, r["test macro-F1"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set(xscale="log", xlabel="parameters (millions, log)", ylabel="test macro-F1",
       title="Accuracy vs. model size")
ax.grid(alpha=0.3)
fig.savefig(OUT / "figures" / "accuracy_vs_size.png", dpi=150, bbox_inches="tight")

## 3. Leakage: patient-level vs image-level split

In [ ]:
leak = table[table["model"].isin(table.loc[table["split"] == "image_split", "model"])]
display(leak[["run", "model", "split", "test accuracy", "test macro-F1"]].round(3))

## 4. Cross-validation summary (mean ± std per model)

In [ ]:
cv = runs[runs["tags.mlflow.runName"].str.contains("fold", na=False)]
if len(cv):
    summary = cv.groupby("tags.model")["metrics.best_val_macro_f1"].agg(["mean", "std", "count"])
    display(summary.round(3))
else:
    print("No CV runs yet")